# Local Invoice OCR on Google Colab
Select **Runtime > Change runtime type > T4 GPU**, reconnect, then run setup cells in order. By default the notebook stops if no GPU is attached. The web app can test PDFs you upload; optional diagnostics below also accept any public-repo PDF or a one-time Colab upload. No Drive upload is needed. Accuracy mode reads original PDF pages with a stock local vision model, then checks item arithmetic and OCR evidence; Fast mode uses spatial OCR only. No training or adapter service is used.

In [ ]:
#@title 1. Clone the public GitHub project
import os, pathlib, subprocess
PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ubaid-148/OCR.git',str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR/'.git').is_dir():
    subprocess.run(['git','-C',str(PROJECT_DIR),'pull','--ff-only'], check=True)
else: raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
os.chdir(PROJECT_DIR)
print('Project ready at', PROJECT_DIR)
print('Flow 2026-09-evidence-gated-v12: stock vision and spatial OCR; uncertain fields stay in review')

In [ ]:
#@title 2. Install dependencies and verify the OCR device
import os, pathlib, shutil, subprocess, sys
REQUIRE_GPU_FOR_ACCURACY = True #@param {type:"boolean"}
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
print('GPU attached:', gpu_runtime, flush=True)
if REQUIRE_GPU_FOR_ACCURACY and not gpu_runtime:
    raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then run all cells again. Accuracy vision on CPU caused a 180-second timeout and must not silently fall back.')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-eng', 'tesseract-ocr-ara', 'tesseract-ocr-urd', 'ghostscript', 'unpaper', 'pngquant', 'zstd'], check=True)
# Keep OCR packages separate from Colab's preinstalled CUDA PyTorch.
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR / 'bin' / 'python')
# pip --python bootstraps pip even in a venv created without it.
ocr_pip = [sys.executable, '-m', 'pip', '--python', OCR_PYTHON]
subprocess.run([*ocr_pip, 'install', '-q', '--upgrade', 'pip'], check=True)
# ModelScope imports torch; its CPU build avoids a second CUDA/NCCL stack.
subprocess.run([*ocr_pip, 'install', '-q', 'torch==2.9.1+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
# CPU and GPU Paddle share a module: install exactly one distribution.
subprocess.run([*ocr_pip, 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=True)
requirements = [line.strip() for line in pathlib.Path('requirements.txt').read_text().splitlines() if line.strip() and not line.strip().startswith('paddlepaddle')]
subprocess.run([*ocr_pip, 'install', '-q', *requirements], check=True)
command = [*ocr_pip, 'install', '-q']
if gpu_runtime:
    command += ['paddlepaddle-gpu==3.3.1', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/']
else:
    command += ['paddlepaddle==3.3.1']
subprocess.run(command, check=True)
# Retry uncertain identifiers/numeric cells with bounded English OCR crops.
os.environ['OCR_TARGETED_RETRY'] = 'true'
os.environ['OCR_DEVICE'] = 'gpu:0' if gpu_runtime else 'cpu'
os.environ['VISION_REQUIRE_GPU'] = 'true'
os.environ.pop('OCR_PYTHON_EXE', None)
# Check in a fresh process so rerunning this cell cannot reuse an old Paddle import.
os.environ.setdefault('FLAGS_use_mkldnn', '0')
verification = subprocess.run(
    [OCR_PYTHON, '-u', str(PROJECT_DIR / 'check_ocr_runtime.py')],
    cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, errors='replace',
)
verification_log = pathlib.Path('/tmp/ocr-runtime-check.log')
verification_log.write_text(verification.stdout, encoding='utf-8')
print(verification.stdout, flush=True)
if verification.returncode:
    raise RuntimeError(
        f'OCR runtime verification failed (exit {verification.returncode}). '
        f'Full log: {verification_log}. Copy the error below:\n\n'
        + verification.stdout[-12000:]
    )
print('Dependencies ready. First upload loads OCR models; later uploads reuse them.')


In [ ]:
#@title 2b. Record the PaddleOCR runtime and model configuration
import json, os, subprocess

probe_code = ("import json,platform,paddle,paddleocr; "
              "print('OCR_RUNTIME_JSON='+json.dumps({'cuda':getattr(paddle,'cuda_version',lambda:None)(),"
              "'gpu_count':paddle.device.cuda.device_count() if paddle.is_compiled_with_cuda() else 0,"
              "'paddle':paddle.__version__,'paddleocr':paddleocr.__version__,'python':platform.python_version()}))")
probe = subprocess.run([OCR_PYTHON, "-c", probe_code], cwd=PROJECT_DIR,
                       capture_output=True, text=True, check=True)
runtime_line = next((line for line in probe.stdout.splitlines() if line.startswith("OCR_RUNTIME_JSON=")), None)
if runtime_line is None:
    raise RuntimeError(f"OCR runtime probe did not return JSON: {probe.stdout} {probe.stderr}")
runtime_info = json.loads(runtime_line.removeprefix("OCR_RUNTIME_JSON="))
gpu_count = runtime_info["gpu_count"]
runtime_device = os.environ.get("OCR_DEVICE", "auto")
if runtime_device == "auto":
    runtime_device = "gpu:0" if gpu_count else "cpu"

try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    ).stdout.strip()
except OSError:
    nvidia_smi = "unavailable"

OCR_MODEL_CONFIGURATION = {
    "GPU": nvidia_smi or ("available" if gpu_count else "not available"),
    "CUDA": runtime_info["cuda"] or "not exposed by Paddle",
    "Paddle": runtime_info["paddle"],
    "PaddleOCR": runtime_info["paddleocr"],
    "Python": runtime_info["python"],
    "Device": runtime_device,
    "Detection model": "PP-OCRv5_mobile_det",
    "English recognition model": "PP-OCRv5_mobile_rec",
    "Arabic recognition model": "arabic_PP-OCRv5_mobile_rec",
}
print(json.dumps(OCR_MODEL_CONFIGURATION, indent=2, ensure_ascii=False))
if runtime_device.startswith("gpu") and not gpu_count:
    print("WARNING: OCR_DEVICE requests GPU but Paddle reports no CUDA device.")


In [ ]:
#@title 3. Start stock vision AI
USE_LOCAL_AI = True #@param {type:"boolean"}
OLLAMA_MODEL = 'qwen3-vl:4b' #@param {type:"string"}
import os, shutil, subprocess, sys, time, urllib.request
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from ollama_http import preload
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
os.environ['USE_LOCAL_AI'] = str(USE_LOCAL_AI).lower()
if USE_LOCAL_AI and not gpu_runtime:
    raise RuntimeError('Vision AI is disabled on CPU for this notebook. Attach a T4/L4 GPU, reconnect, and rerun from cell 1; or set USE_LOCAL_AI=False for spatial-only Fast mode.')
os.environ['OLLAMA_TIMEOUT_SECONDS'] = '180'
os.environ['OLLAMA_NUM_CTX'] = '16384'
os.environ['OLLAMA_NUM_PREDICT'] = '4096'
os.environ.pop('OLLAMA_URL', None)
if USE_LOCAL_AI:
    if not shutil.which('ollama'):
        ollama_archive = '/tmp/ollama-linux-amd64.tar.zst'
        print('Downloading Ollama...')
        urllib.request.urlretrieve('https://ollama.com/download/ollama-linux-amd64.tar.zst', ollama_archive)
        subprocess.run(['tar', '--zstd', '-xf', ollama_archive, '-C', '/usr'], check=True)
        if not shutil.which('ollama'):
            raise RuntimeError('Ollama archive extracted but the executable was not found')
    # Cell 1 replaces /content/OCR. Restart Ollama so it never retains that deleted cwd.
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as response:
            service_running = response.status == 200
    except OSError:
        service_running = False
    if service_running:
        subprocess.run(['ollama', 'stop', OLLAMA_MODEL], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        subprocess.run(['pkill', '-TERM', '-x', 'ollama'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        time.sleep(2)
    if 'ollama_log' in globals() and not ollama_log.closed:
        ollama_log.close()
    ollama_log = open('/tmp/ollama.log', 'a')
    ollama_process = subprocess.Popen(
        ['ollama', 'serve'], cwd='/content', start_new_session=True,
        stdout=ollama_log, stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start. Check /tmp/ollama.log')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    print('Loading AI model before the first invoice...')
    ai_preloaded = preload(OLLAMA_MODEL, context=int(os.environ['OLLAMA_NUM_CTX']))
    if not ai_preloaded:
        raise RuntimeError('Vision model did not preload. Check /tmp/ollama.log; the app will not launch into a slow spatial fallback.')
    placement = subprocess.run(['ollama', 'ps'], capture_output=True, text=True, check=True)
    print(placement.stdout, flush=True)
    model_line = next((line for line in placement.stdout.splitlines()[1:] if line.split() and line.split()[0] == OLLAMA_MODEL), '')
    if '100% GPU' not in model_line:
        raise RuntimeError(f'{OLLAMA_MODEL} is not fully on GPU. Ollama placement: {model_line or placement.stdout}. Use a runtime with more free GPU memory; CPU offload is too slow for Accuracy mode.')
    print('Ollama setup complete: vision model preloaded fully on GPU.')
else:
    os.environ['OLLAMA_URL'] = 'http://127.0.0.1:1/api/chat'
    print('Local AI disabled; deterministic spatial fallback will be used.')

In [ ]:
#@title 4. Start OCR web application
import os, subprocess, sys, time, urllib.request
os.chdir('/content/OCR')
if 'OCR_PYTHON' not in globals():
    raise RuntimeError('Run dependency setup (cell 2) first.')
if 'ocr_process' in globals() and ocr_process.poll() is None:
    ocr_process.terminate()
    ocr_process.wait(timeout=15)
os.environ['OCR_PRELOAD'] = 'true'
print('Preparing OCR models before accepting uploads; first startup may download models.')
ocr_log = open('/tmp/ocr-web.log', 'w')
ocr_process = subprocess.Popen([OCR_PYTHON, '-u', 'ocr_web.py'], stdout=ocr_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for attempt in range(600):
    if attempt and attempt % 15 == 0:
        print('Still preparing OCR models. Recent log:', open('/tmp/ocr-web.log').read()[-600:])
    if ocr_process.poll() is not None:
        print(open('/tmp/ocr-web.log').read())
        raise RuntimeError('OCR server exited during startup')
    try:
        response = urllib.request.urlopen('http://127.0.0.1:8765/', timeout=2)
        if response.status == 200:
            break
    except Exception:
        time.sleep(1)
else:
    print(open('/tmp/ocr-web.log').read())
    raise RuntimeError('OCR web application did not start')
print('OCR application is ready.')

In [ ]:
#@title 5. Open the application
from google.colab import output
output.serve_kernel_port_as_iframe(8765, height='700')

In [ ]:
#@title 6. Choose any PDF and optionally run raw PaddleOCR evidence
PDF_SOURCE = "repo" #@param ["repo", "upload"]
PDF_NAME = "" #@param {type:"string"}
TEST_PAGE = 1 #@param {type:"integer"}
RUN_RAW_BENCHMARK = False #@param {type:"boolean"}
import hashlib, json, os, re, subprocess, tempfile
from pathlib import Path

sample_path = None
benchmark_dir = None
raw_ocr, parser_result, parser_view = None, None, None
raw_rows, numeric_rows = [], []
if PDF_SOURCE == "repo" and PDF_NAME.strip():
    public_dir = (PROJECT_DIR / "public_invoice_pdfs").resolve()
    candidate = (public_dir / PDF_NAME.strip()).resolve()
    if not candidate.is_relative_to(public_dir):
        raise ValueError("Select a PDF inside public_invoice_pdfs, not a parent path.")
    sample_path = candidate
elif PDF_SOURCE == "upload":
    from google.colab import files
    upload_dir = Path(tempfile.mkdtemp(prefix="invoice-benchmark-"))
    previous_cwd = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one PDF.")
    sample_path = upload_dir / next(iter(uploaded))
elif PDF_SOURCE != "repo":
    raise ValueError(f"Unknown PDF source: {PDF_SOURCE}")

if sample_path is not None:
    if not sample_path.is_file() or sample_path.suffix.lower() != ".pdf":
        raise ValueError(f"PDF not found: {sample_path}")
    with sample_path.open("rb") as pdf_file:
        if pdf_file.read(5) != b"%PDF-":
            raise ValueError(f"Not a PDF file: {sample_path}")
        digest = hashlib.sha256(b"%PDF-")
        for chunk in iter(lambda: pdf_file.read(1024 * 1024), b""):
            digest.update(chunk)
    safe_stem = re.sub(r"[^A-Za-z0-9_-]+", "_", sample_path.stem)[:60] or "invoice"
    benchmark_dir = PROJECT_DIR / "benchmark_outputs" / f"{safe_stem}-{digest.hexdigest()[:10]}"
    benchmark_dir.mkdir(parents=True, exist_ok=True)
    print("Selected PDF:", sample_path, "| evidence directory:", benchmark_dir)
else:
    available = sorted(path.name for path in (PROJECT_DIR / "public_invoice_pdfs").glob("*.pdf"))
    print("Enter PDF_NAME (a repo filename) or choose PDF_SOURCE=upload. Available:", available[:30])

if RUN_RAW_BENCHMARK:
    if sample_path is None:
        raise ValueError("Choose a PDF in this cell before running the raw benchmark.")
    if "OCR_PYTHON" not in globals():
        raise RuntimeError("Run dependency setup (cell 2) first.")
    subprocess.run([OCR_PYTHON, "-u", "-m", "tools.ocr_diagnostics", "raw",
                    "--pdf", str(sample_path), "--output-dir", str(benchmark_dir)],
                   cwd=PROJECT_DIR, check=True)
    raw_ocr = json.loads((benchmark_dir / "raw_paddleocr.json").read_text(encoding="utf-8"))
    parser_result = json.loads((benchmark_dir / "parser_full.json").read_text(encoding="utf-8"))
    parser_view = json.loads((benchmark_dir / "parser_fast.json").read_text(encoding="utf-8"))
    raw_rows = [word for page in raw_ocr.get("pages", []) for word in page.get("words", [])]
    numeric_rows = [word for word in raw_rows if re.search(r"\d", str(word.get("text", "")))]
else:
    print("Raw benchmark disabled. Select a PDF and set RUN_RAW_BENCHMARK=True to run it.")


In [ ]:
#@title 7. Compare PaddleOCR preprocessing variants
RUN_PREPROCESSING_COMPARISON = False #@param {type:"boolean"}
comparison = []

if RUN_PREPROCESSING_COMPARISON:
    if sample_path is None:
        raise ValueError("Choose a PDF in cell 6 before preprocessing comparison.")
    subprocess.run([OCR_PYTHON, "-u", "-m", "tools.ocr_diagnostics", "preprocess",
                    "--pdf", str(sample_path), "--output-dir", str(benchmark_dir),
                    "--page", str(TEST_PAGE)], cwd=PROJECT_DIR, check=True)
    preprocessing_path = benchmark_dir / f"page-{TEST_PAGE}-preprocessing-comparison.json"
    comparison = json.loads(preprocessing_path.read_text(encoding="utf-8"))
else:
    print("Preprocessing comparison disabled.")


In [ ]:
#@title 8. Draw readable OCR boxes for table inspection
if RUN_RAW_BENCHMARK and raw_ocr and raw_ocr.get("pages"):
    from IPython.display import Image as NotebookImage, display
    subprocess.run([OCR_PYTHON, "-u", "-m", "tools.ocr_diagnostics", "annotate",
                    "--pdf", str(sample_path), "--output-dir", str(benchmark_dir),
                    "--page", str(TEST_PAGE)], cwd=PROJECT_DIR, check=True)
    annotated_path = benchmark_dir / f"page-{TEST_PAGE}-ocr-boxes.png"
    display(NotebookImage(filename=str(annotated_path)))
    print("Saved annotated OCR evidence to", annotated_path)
else:
    print("Run the raw benchmark first to create the annotated page.")


In [ ]:
#@title 9. Generate the OCR-versus-parser benchmark report
REFERENCE_JSON = "" #@param {type:"string"}
if not RUN_RAW_BENCHMARK or raw_ocr is None:
    print("Report skipped; run the raw benchmark in cell 6 first.")
else:
    report = {
        "input": str(sample_path),
        "ocr_engine": "PaddleOCR",
        "ocr_runtime": globals().get("OCR_MODEL_CONFIGURATION", {}),
        "raw_ocr": {
            "pages": len(raw_ocr.get("pages", [])),
            "detected_boxes": len(raw_rows),
            "numeric_boxes": len(numeric_rows),
            "elapsed_seconds": raw_ocr.get("benchmark", {}).get("elapsed_seconds"),
            "evidence_file": str(benchmark_dir / "raw_paddleocr.json"),
        },
        "parser": parser_view,
        "preprocessing": comparison,
        "interpretation": [
            "Raw OCR evidence is stored with page, text, confidence, and bounding box.",
            "Parser output is generated from the same OCR pages and is shown separately.",
            "No invoice value is used as an expected pass condition.",
            "A raw text/box error indicates OCR or preprocessing work; a correct raw box mapped to a wrong item indicates table/parser association work.",
        ],
    }
    report_path = benchmark_dir / "benchmark_report.json"
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print("Report saved to", report_path)
    if REFERENCE_JSON.strip():
        from tools.attribute_errors import attribute, report as attribution_report
        reference_path = Path(REFERENCE_JSON.strip()).expanduser()
        if not reference_path.is_absolute():
            reference_path = PROJECT_DIR / reference_path
        reference = json.loads(reference_path.read_text(encoding="utf-8"))
        if reference.get("source_filename") not in (None, sample_path.name):
            raise ValueError("Reference source_filename does not match the selected PDF.")
        attribution = attribute(sample_path, reference, raw_ocr, parser_result)
        attribution_path = benchmark_dir / "error_attribution.json"
        attribution_path.write_text(json.dumps(attribution, ensure_ascii=False, indent=2), encoding="utf-8")
        print(attribution_report(attribution))
        print("Attribution saved to", attribution_path)
        if reference.get("source_filename") is None:
            print("Reference has no source_filename binding; confirm it belongs to this PDF before trusting comparisons.")
        if attribution["unresolved"]:
            print("Unresolved fields need verified source bboxes; counts are not a complete root-cause verdict.")
    else:
        print("No reference supplied; ground-truth attribution skipped.")


For the app, run cells 1-5. For diagnostics, use cell 6 to select **any** public-repo PDF by filename (for example `9481.pdf`) or choose `upload` for a private runtime-only PDF; set `RUN_RAW_BENCHMARK=True` to collect raw OCR. Cells 7-9 are optional and use that same selected PDF. The Stage 1 sweep below is OFF by default because it makes 12 extra OCR passes: run cell 6 to select a PDF, choose `TEST_PAGE`, then set `RUN_STAGE1_SWEEP=True`. It saves exact parameters and timings without changing production defaults. Do not interpret a parameter sweep as a ground-truth accuracy score. A comparison reference is used only if you explicitly set `REFERENCE_JSON` in cell 9, and must belong to the selected PDF.

In [ ]:
#@title 10. Optional Stage 1 OCR DPI/side-limit sweep (12 passes)
RUN_STAGE1_SWEEP = False #@param {type:"boolean"}
import subprocess
if RUN_STAGE1_SWEEP:
    if sample_path is None or benchmark_dir is None:
        raise ValueError('Select a PDF in cell 6 before running Stage 1.')
    command = [OCR_PYTHON, '-u', '-m', 'tools.ocr_param_sweep',
               '--pdf', str(sample_path),
               '--page', str(TEST_PAGE), '--output-dir', str(benchmark_dir)]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
else:
    print('Stage 1 sweep skipped. Set RUN_STAGE1_SWEEP=True and run this cell after setup.')
